# Eksik Gözlem Çözümü
Eksik değerlerin türünü (MCAR/MAR/MNAR) belirledikten sonra, şimdi 
**"bu eksiklikle ne yapacağız"** sorusuna cevap veriyoruz. Yöntem seçimi, 
büyük ölçüde eksikliğin **türüne** ve **oranına** bağlıdır.

## 1. Silme Yöntemleri (Deletion)

**a) Listwise Deletion (Satır Silme):** Herhangi bir kolonu eksik olan **tüm satırı** siler.
```python
df_temiz = df.dropna()
```
- ✅ Basit
- ❌ Veri kaybı fazla olabilir, özellikle çok kolon varsa
- ⚠️ Sadece **MCAR** ise güvenlidir — MAR/MNAR'da **yanlılık (bias)** 
  yaratabilir (çünkü eksik olan satırlar rastgele değil, belirli bir 
  gruba ait olabilir, onları silmek o grubu az temsil etmene sebep olur)

**b) Pairwise Deletion:** Her analiz için, sadece o analizde gereken 
kolonlarda eksik olan satırları siler (diğer analizlerde farklı satırlar 
kullanılabilir). Daha az veri kaybettirir ama karmaşıklık yaratabilir.

## 2. Doldurma Yöntemleri (Imputation)

**a) Basit İstatistiksel Doldurma:**
```python
df['Yas'].fillna(df['Yas'].mean(), inplace=True)    # ortalama ile
df['Yas'].fillna(df['Yas'].median(), inplace=True)  # medyan ile (çarpık veri için daha güvenli, Konu 6-7'yi hatırla)
df['Kategori'].fillna(df['Kategori'].mode()[0], inplace=True)  # kategorik için mod
```
- ⚠️ Değişkenin **varyansını yapay olarak azaltır** (hepsi aynı değere 
  dolduğu için) — dikkatli kullanılmalı

**b) İleri Düzey Doldurma (İsimlerini Bilmen Yeterli):**
- **KNN Imputation:** Eksik değeri, "en benzer" diğer gözlemlerin 
  ortalamasıyla doldurur
- **Regresyon ile Doldurma:** Eksik değeri, diğer değişkenlerden 
  regresyon (Konu 113+) ile tahmin eder
- **Multiple Imputation (MICE):** Eksik değeri **birden fazla kez, 
  farklı olasılıklarla** doldurup sonuçları birleştirir — MAR verilerde 
  altın standart kabul edilir

## Karar Ağacı (Basitleştirilmiş)

- Eksiklik oranı çok düşükse (%1-2) → Listwise deletion genelde güvenli.
- Eksiklik MCAR ise → Silme veya basit doldurma kabul edilebilir.
- Eksiklik MAR ise → Regresyon/KNN/MICE gibi daha akıllı yöntemler tercih edilmeli.
- Eksiklik MNAR ise → En zor durum, domain bilgisiyle özel çözüm gerekir,basit yöntemler ciddi yanlılık yaratabilir.


## Neden Bu Kadar Önemli?
Yanlış eksik değer yönetimi, ileride kuracağımız istatistiksel testlerin ve ML modellerinin **sonuçlarını sessizce bozabilir** — bu, veri biliminde en çok "gözden kaçan ama en çok zarar veren" hatalardan biridir.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 300

yas = np.random.normal(40, 12, n)
gelir = 2000 + yas * 80 + np.random.normal(0, 800, n)
harcama = gelir * 0.3 + np.random.normal(0, 300, n)

df = pd.DataFrame({'Yas': yas, 'Gelir': gelir, 'Harcama': harcama})
gelir_yas_korelasyon = df[['Yas', 'Gelir']].corr()
print(gelir_yas_korelasyon)
# MAR kurgusu: Genç müşteriler (Yas < 30), Gelir bilgisini paylaşmaya daha az istekli
eksik_olasiligi = np.where(df['Yas'] < 30, 0.6, 0.05)
eksik_mask = np.random.random(n) < eksik_olasiligi
df.loc[eksik_mask, 'Gelir'] = np.nan

            Yas     Gelir
Yas    1.000000  0.765216
Gelir  0.765216  1.000000


In [2]:
df['Gelir_Eksik'] = df['Gelir'].isnull().astype(int)
df

,Yas,Gelir,Harcama,Gelir_Eksik
0,45.960570,5013.649578,1731.191458,0
1,38.340828,4619.121439,1109.086834,0
2,47.772262,6419.615881,2186.766540,0
3,58.276358,7150.404874,2551.812820,0
4,37.190160,NaN,1611.577916,1
...,...,...,...,...
295,31.685085,4126.793669,1432.651067,0
296,50.795199,5847.715932,1704.179356,0
297,43.687594,4711.996567,1457.613076,0
298,49.754345,5624.913026,2049.426598,0


In [3]:
df['Yas_Grubu'] = pd.cut(x=df['Yas'], bins=[0,30,100], labels=['30 yaş altı', '30 yaş üstü'])
df

,Yas,Gelir,Harcama,Gelir_Eksik,Yas_Grubu
0,45.960570,5013.649578,1731.191458,0,30 yaş üstü
1,38.340828,4619.121439,1109.086834,0,30 yaş üstü
2,47.772262,6419.615881,2186.766540,0,30 yaş üstü
3,58.276358,7150.404874,2551.812820,0,30 yaş üstü
4,37.190160,NaN,1611.577916,1,30 yaş üstü
...,...,...,...,...,...
295,31.685085,4126.793669,1432.651067,0,30 yaş üstü
296,50.795199,5847.715932,1704.179356,0,30 yaş üstü
297,43.687594,4711.996567,1457.613076,0,30 yaş üstü
298,49.754345,5624.913026,2049.426598,0,30 yaş üstü


In [4]:
from scipy import stats
kontenjans = pd.crosstab(df['Yas_Grubu'], df['Gelir_Eksik'])
print(kontenjans)

Gelir_Eksik    0   1
Yas_Grubu           
30 yaş altı   25  35
30 yaş üstü  226  14


In [5]:
# H0: Yaş Grubu ve Gelirin Eksik olması arasında anlamlı bir ilişki yoktur.
# H1: Yaş Grubu ve Gelirin Eksik olması arasında anlamlı bir ilişki vardır.
chi2, p, dof, beklenen = stats.chi2_contingency(observed=kontenjans)
print(f'p-değeri: {p}')
alpha = 0.05
if p < alpha: # type: ignore
    print('Yaş Grubu ve Gelirin eksik olması arasında istatistiksel olarak anlamlı bir ilişki vardır.')
else:
    print('Yaş Grubu ve Gelirin eksik olması arasında istatistiksel olarak anlamlı bir ilişki yoktur.')

p-değeri: 5.205481208963786e-22
Yaş Grubu ve Gelirin eksik olması arasında istatistiksel olarak anlamlı bir ilişki vardır.


In [6]:
ortalama_gelir = df['Gelir'].mean()
df['Yeni_Gelir'] = df.groupby('Yas_Grubu')['Gelir'].transform(lambda x: x.fillna(x.mean()))
df['Gelir'] = df['Gelir'].fillna(value=ortalama_gelir)
naif_korelasyon = df[['Gelir', 'Yas']].corr()
print(naif_korelasyon)
grup_bazlı_korelasyon = df[['Yeni_Gelir', 'Yas']].corr()
print(grup_bazlı_korelasyon)

          Gelir       Yas
Gelir  1.000000  0.609652
Yas    0.609652  1.000000
            Yeni_Gelir       Yas
Yeni_Gelir    1.000000  0.758435
Yas           0.758435  1.000000


C:\Users\asus\AppData\Local\Temp\ipykernel_9264\931230273.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df['Yeni_Gelir'] = df.groupby('Yas_Grubu')['Gelir'].transform(lambda x: x.fillna(x.mean()))


### Sonuç
Öncelikle yapay bir veri oluşturduk. Bu veri setinde, gelir yaş arttıkça artacak şekilde kurgulandı. Ek olarak, genç müşteriler gelir bilgilerini paylaşmaya daha az istekli şeklinde bir yapı oluşturduk. Bu yapı neticesinde, gelir sütununa ait boş gözlemlerin yaş ile ilişkisi de olmuş oldu (MAR). Ardından gelir sütununa ait boş gözlemleri yaş gruplarını ayırmaksızın tek bir ortalama ile doldurunca yaş arttıkça gelirin de arttığı bir düzendeki korelasyon katsayısı düşmüş oldu çünkü yaş arttıkça gelirin de arttığı yapı biraz bozuldu. En sonda ise daha doğru bir yöntem olan yaş gruplarına göre gelir sütunundaki boş gözlemleri doldurma yöntemiyle 0.758435 gibi ilk korelasyon katsayısına (0.765216) yaklaşık bir r katsayısı elde ettik.

In [ ]:
np.random.seed(42)
n=300
yas = np.random.normal(loc=40, scale=12, size=n)
aylik_ziyaret_sayisi = np.random.randint(low=1, high=10, size=n)
yillik_harcama = (2000) + (aylik_ziyaret_sayisi * 100) + np.random.normal(0,150,size=n)
